In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [4]:
df=pd.read_csv('/Users/haydenmonk/Downloads/HPCC/Mass_Fraction/Outputs/Ejection_Results/2249453_ejection_results.txt', header=None)

In [5]:
df.columns=['mass','time','result','particle_id']

In [6]:
def compute_cumulative_outcome_fractions(df, masses, N):
    """
    df      : DataFrame with columns ['mass', 'time', 'particle_id', 'result']
              where result is 'Ejected' or 'Captured', and each row represents
              the time at which that particle's outcome occurred
    masses  : list/array of unique mass values, in the same order as N
    N       : list/array of total particle counts, one per mass in `masses`

    Returns
    -------
    ejection_df, capture_df : DataFrames indexed by mass, columns = sorted
                               unique time values, entries = CUMULATIVE
                               fraction of N with that result by that time
                               (i.e. fraction ejected/captured at or before
                               each time, monotonically non-decreasing
                               across each row)
    """
    if len(masses) != len(N):
        raise ValueError("masses and N must be the same length")

    mass_to_N = dict(zip(masses, N))

    times = np.sort(df["time"].unique())

    ejection_df = pd.DataFrame(index=masses, columns=times, dtype=float)
    capture_df = pd.DataFrame(index=masses, columns=times, dtype=float)

    for mass in masses:
        mass_df = df[df["mass"] == mass]
        denom = mass_to_N[mass]

        counts = (
            mass_df.groupby(["time", "result"])
            .size()
            .unstack(fill_value=0)
        )

        # reindex to the full sorted time axis, filling any missing times
        # with 0 new outcomes at that time, then cumsum across time
        ejected_counts = counts.get("Ejected", pd.Series(0, index=counts.index))
        captured_counts = counts.get("Captured", pd.Series(0, index=counts.index))

        ejected_counts = ejected_counts.reindex(times, fill_value=0)
        captured_counts = captured_counts.reindex(times, fill_value=0)

        ejection_df.loc[mass] = (ejected_counts.cumsum() / denom).values
        capture_df.loc[mass] = (captured_counts.cumsum() / denom).values

    return ejection_df, capture_df

In [7]:
df.mass.unique()

array([5.01187234e-03, 2.51188643e-03, 1.00000000e-02, 3.16227766e-04,
       6.30957344e-04, 1.25892541e-03, 1.58489319e-04, 7.94328235e-05])

In [8]:
ejection_df,c=compute_cumulative_outcome_fractions(df,df.mass.unique(),[800,800,800,800,800,800,800,800])

In [11]:
ejection_df.sort_index(ascending=False)

,0.000000e+00,9.998419e+02,4.993732e+03,4.998423e+03,1.197004e+04,1.691563e+04,1.791067e+04,1.890571e+04,1.990074e+04,2.089578e+04,...,3.793699e+06,3.870693e+06,3.959686e+06,4.213666e+06,4.238664e+06,4.242664e+06,4.293660e+06,4.886613e+06,5.053600e+06,5.119594e+06
0.010000,0.0,0.0,0.0,0.0,0.0,0.0025,0.0125,0.01625,0.02125,0.02625,...,0.46125,0.46125,0.46125,0.46125,0.46125,0.46125,0.46125,0.46125,0.46125,0.46125
0.005012,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.00000,0.00000,0.00000,...,0.41875,0.41875,0.41875,0.41875,0.41875,0.41875,0.41875,0.41875,0.41875,0.41875
0.002512,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.00000,0.00000,0.00000,...,0.30625,0.30625,0.30625,0.30625,0.30625,0.30625,0.30625,0.30625,0.30625,0.30625
0.001259,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.00000,0.00000,0.00000,...,0.23500,0.23500,0.23500,0.23500,0.23500,0.23500,0.23500,0.23500,0.23500,0.23500
0.000631,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.00000,0.00000,0.00000,...,0.21500,0.21500,0.21500,0.21500,0.21500,0.21500,0.21500,0.21500,0.21500,0.21500
0.000316,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.00000,0.00000,0.00000,...,0.18875,0.18875,0.18875,0.18875,0.18875,0.18875,0.18875,0.18875,0.18875,0.18875
0.000158,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.00000,0.00000,0.00000,...,0.15125,0.15250,0.15375,0.15500,0.15625,0.15750,0.15875,0.16000,0.16125,0.16250
0.000079,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.00000,0.00000,0.00000,...,0.05625,0.05625,0.05625,0.05625,0.05625,0.05625,0.05625,0.05625,0.05625,0.05625
